In [ ]:
# !/usr/bin/env python3
#coding=utf-8
import time
import binascii
import socket
import struct
import sys
from Arm_Lib import Arm_Device
import select
import torch
import torchvision
import cv2
import ipywidgets.widgets as widgets
import threading
import time
import enum
import colorsys
import numpy as np

# Fix Address already in use
import os
os.system('fuser -kn tcp 65432 | awk')


image_widget = widgets.Image(format='jpeg', width=512, height=512)  #Set up camera display components
display(image_widget)                                               #Display camera assembly

cam = cv2.VideoCapture(0)                           #Open camera

print("torch cuda is available:", torch.cuda.is_available())
model = torch.load("cube_model_500.pth", map_location=torch.device('cuda'))

def bgr8_to_jpeg(value, quality=75):
    if value is None:
        print(value)
        return None
    return bytes(cv2.imencode('.jpg', value)[1])
   


def random_colours(N, enable_random=True, num_channels=3):
    """
    Generate random colors.
    Generate visually distinct colours by linearly spacing the hue 
    channel in HSV space and then convert to RGB space.
    """
    start = 0
    if enable_random:
        random.seed(10)
        start = random.random()
    hues = [(start + i / N) % 1.0 for i in range(N)]
    colours = [list(colorsys.hsv_to_rgb(h, 0.9, 1.0)) for i, h in enumerate(hues)]
    if num_channels == 4:
        for color in colours:
            color.append(1.0)
    if enable_random:
        random.shuffle(colours)
    return colours


def draw_boxes(image, bboxes, labels=None, colours=None, label_size=10):
    if colours is None:
        colours = random_colours(len(bboxes))
    if labels is None:
        labels = [""] * len(bboxes)
    for bb, label, colour in zip(bboxes, labels, colours):
        maxint = 2 ** (struct.Struct("i").size * 8 - 1) - 1
        # if a bbox is not visible, do not draw
        if bb[0] != maxint and bb[1] != maxint:
            start_point = (int(bb[0]),int(bb[3]))
            end_point = (int(bb[2]), int(bb[1]))
            color = (0,255,0)
            
            image = cv2.rectangle(image, start_point, end_point, color, 3)

    # show the image        
    image_widget.value = bgr8_to_jpeg(image)
    

def run_inferencing(model, image):
    
    torch_image = torch.tensor(image, dtype=torch.float, device="cuda")
    torch_image = torch_image.float() / 255.0
    torch_image = torch_image.permute(2, 0, 1)
    
    model.eval()
    with torch.no_grad():
        images=[torch_image, torch_image] 
        predictions = model(images[:1])
        #print(predictions)
                
        idx = 0
        score_thresh = 0.7

        pred = predictions[idx]

        np_image = images[idx].permute(1, 2, 0).cpu().numpy()
        score_filter = [i for i in range(len(pred["scores"])) if pred["scores"][i] > score_thresh]
        num_instances = len(score_filter)
        colours = random_colours(num_instances, enable_random=False)
        #print("num_intances ", num_instances, " ", colours)
        
        categories = ["None", "Cube", "Sphere", "Cone"]
        mapping = {i + 1: cat for i, cat in enumerate(categories)}
        labels = [mapping[label.item()] for label in pred["labels"]]

        draw_boxes(image, pred["boxes"].tolist(), labels=labels, colours=colours, label_size=10)
        

HOST = ''  # Standard loopback interface address (localhost)
PORT = 65432        # Port to listen on (non-privileged ports are > 1023)

# Get DOFBOT object
Arm = Arm_Device()
time.sleep(.1)
base_id = 1
shoulder_pan_id = 2
shoulder_lift_id = 3
elbow_id = 4
wrist_id = 5  
finger_id = 6
sleep_time_inbetween = 0.08

def receiveTextViaSocket(sock):
    unpacker = struct.Struct('f f f f f f')

    connection, client_address = sock.accept()
    data = connection.recv(unpacker.size)
    stringifiedData = str(binascii.hexlify(data))
    print("received" + stringifiedData)

    ACK_TEXT = 'text_not_received'
    if stringifiedData != "b''":
        unpacked_data = unpacker.unpack(data)
        print("unpacked:", str(unpacked_data))
        Arm.Arm_serial_servo_write(base_id, unpacked_data[base_id - 1], 500)
        time.sleep(sleep_time_inbetween)
        Arm.Arm_serial_servo_write(shoulder_pan_id, unpacked_data[shoulder_pan_id - 1], 300)
        time.sleep(sleep_time_inbetween)
        Arm.Arm_serial_servo_write(shoulder_lift_id, unpacked_data[shoulder_lift_id - 1], 300)
        time.sleep(sleep_time_inbetween)
        Arm.Arm_serial_servo_write(elbow_id, unpacked_data[elbow_id - 1], 300)
        time.sleep(sleep_time_inbetween)
        Arm.Arm_serial_servo_write(wrist_id, unpacked_data[wrist_id - 1], 300)
        time.sleep(sleep_time_inbetween)
        Arm.Arm_serial_servo_write(finger_id, unpacked_data[finger_id - 1], 300)
        time.sleep(sleep_time_inbetween)
        ACK_TEXT = 'text_received'

    encodedAckText = bytes('complete', 'utf-8')
    # send the encoded acknowledgement text
    connection.sendall(encodedAckText)
    connection.close()

def new_server():
    # Create a TCP/IP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_address = (HOST, PORT)
    sock.bind(server_address)
    sock.listen(1)

    unpacker = struct.Struct('f f f f f f')
    socks = [sock]
    counter = 0
    while True:
        print('waiting for a connection')
        
        readySocks, _, _ = select.select(socks, [], [], 5)
        for sock in readySocks:
            receiveTextViaSocket(sock)
        

        ret, frame = cam.read()
        if frame is not None :
            run_inferencing(model, frame)

new_server()

In [ ]:
del Arm  # Release DOFBOT object